<a href="https://colab.research.google.com/github/cook1e-0707/practicalAI-lab/blob/main/day2/student/week1_day2_yourname.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/>
</a>

# Week 1 · Day 2 Student Lab
## Supervised Learning with Four Regression Models

**Before you start:** Choose **File → Save a copy in Drive**.
Rename it `week1_day2_yourname.ipynb`.


## Today

Day 1 introduced data, training, prediction, and Linear Regression.
Today we ask a new question:

> What changes when four different models learn from the same data?

We will use one lemonade-sales problem with four models:

1. Linear Regression
2. Decision Tree
3. K-Nearest Neighbors (KNN)
4. A small Neural Network

Every model will use the same data and the same steps:

```text
X_train + y_train → .fit() → trained model
X_validation → .predict() → validation predictions
validation predictions + y_validation → MAE
```

The model changes, but the supervised-learning process stays the
same.

You do not need to memorize four algorithms or every line of
plotting code. Focus on:

1. how each model makes a prediction;
2. what stays the same across all four models;
3. how validation MAE lets us compare them fairly.


# Supervised Learning

In **supervised learning**, every training example has:

- **features**: information given to the model;
- a **target**: the answer we want the model to learn to predict.

Supervised learning can predict:

- a **number**: regression;
- a **category**: classification.

Today we predict a number, `cups_sold`, so all four models are
used for **regression**.

| Model | Main prediction idea | Shape we may see |
|---|---|---|
| Linear Regression | one weighted rule | a line |
| Decision Tree | a sequence of questions | steps |
| KNN | nearby training examples | local averages |
| Neural Network | learned connections between layers | a flexible curve |

`.fit(...)` does **not** mean that every model changes in the
same way:

| Model | What happens inside `.fit(...)` |
|---|---|
| Linear Regression | calculate and store the best coefficients and intercept |
| Decision Tree | choose questions and grow branches |
| KNN | scale and store the training examples |
| Neural Network | repeatedly adjust connection weights |


## Training, Validation, and Test Data

- **Training data:** the model learns from it.
- **Validation data:** we compare models and make choices.
- **Test data:** we check the final choice once at the end.

```text
Training data   → learn
Validation data → choose
Test data       → final check
```

**Think before continuing:** Which data should stay closed until
the end?


# Load and View the Data

This is a **synthetic teaching dataset** with 72 fictional
lemonade-stand days. It includes random variation so the models
will not make perfect predictions.

| Column | Meaning |
|---|---|
| `temperature_f` | temperature in degrees Fahrenheit |
| `weekend` | `1` for a weekend, otherwise `0` |
| `rain` | `1` for rain, otherwise `0` |
| `cups_sold` | number of cups sold |

We import Pandas here because this is the first time we use it
today.


In [ ]:
import pandas as pd

print("Imported package:", pd.__name__)
print("Package version:", pd.__version__)


## Task 1 — Read the CSV File

Choose one way to get the CSV:

1. Use the GitHub raw-data URL.
2. Upload the CSV to Colab and enter its filename.
3. Mount Google Drive and enter the Drive path.

Put your chosen URL or path in `data_source`, then use
`pd.read_csv(...)`.


In [ ]:
DATA_URL = (
    "https://raw.githubusercontent.com/cook1e-0707/"
    "practicalAI-lab/main/day2/data/lemonade_sales.csv"
)

data_source = ""  # TODO: enter DATA_URL, a filename, or a Drive path
data = None       # TODO: read data_source with Pandas

data.head()


<details>
<summary>Hint</summary>

For the GitHub method, set `data_source = DATA_URL`. Then use:

```python
data = pd.read_csv(data_source)
```
</details>


## Task 2 — Explore the Table

Find:

- the number of rows and columns;
- the average number of cups sold;
- the number of rainy days;
- whether any values are missing.


In [ ]:
print("Rows and columns:", data.shape)

average_cups = None  # TODO: mean of cups_sold
rainy_days = None    # TODO: keep rows where rain equals 1

print("Average cups sold:", average_cups)
print("Number of rainy days:", len(rainy_days))
print("\nMissing values:")
print(data.isna().sum())


<details>
<summary>Hint</summary>

Use `.mean()` on one column. Filter rainy rows with:

```python
data[data["column_name"] == value]
```
</details>


## First Picture of the Data

We import Matplotlib here because this is the first chart today.

`plt.scatter(x_values, y_values)` draws one point for each pair
of values. We use it because a picture can reveal a relationship
before we train a model. We draw dry and rainy days separately
so the legend explains the colors.

Each point is one day. The points show a general pattern, but
they do not make one perfect line.


In [ ]:
import matplotlib.pyplot as plt

dry_days = data[data["rain"] == 0]
rainy_days = data[data["rain"] == 1]

plt.scatter(
    dry_days["temperature_f"],
    dry_days["cups_sold"],
    label="No rain",
    color="#1976D2",
    alpha=0.8,
)
plt.scatter(
    rainy_days["temperature_f"],
    rainy_days["cups_sold"],
    label="Rain",
    color="#EF6C00",
    alpha=0.8,
)
plt.title("Temperature and lemonade sales")
plt.xlabel("Temperature (°F)")
plt.ylabel("Cups sold")
plt.legend()
plt.show()


**Look at the chart:**

1. What does one point represent?
2. Does higher temperature usually go with higher sales?
3. Do all days with the same temperature have the same sales?
4. What other columns might help explain the differences?


# Features, Target, and MAE

We will give every model the same three features:

- `temperature_f`
- `weekend`
- `rain`

The target is `cups_sold`.


## Task 3 — Create `X` and `y`

In scikit-learn examples:

- `X` usually means the feature table;
- `y` usually means the target column.


In [ ]:
feature_columns = ["temperature_f", "weekend", "rain"]
target_column = None  # TODO: enter the target column name

X = data[feature_columns]
y = data[target_column]

print("Feature columns:", list(X.columns))
print("Target:", y.name)


## Task 4 — Split the Data

`train_test_split(X, y, test_size=...)` divides matching
feature rows and target answers without separating them. We use
it so each feature row stays connected to its correct answer
while the data is divided.

We call it twice:

1. separate training data from the remaining data;
2. divide the remaining data into validation and test data.

The provided code creates approximately:

- 60% training data;
- 20% validation data;
- 20% test data.

Before splitting, `train_test_split(...)` normally shuffles the
rows. Shuffling helps prevent the original row order from
deciding which rows become training, validation, or test data.

The shuffle is **pseudo-random**: it looks random, but a starting
number called a **seed** can make it repeat the same order.

`random_state=42` sets that seed:

- the same data, code, and seed produce the same split again;
- a different seed usually produces a different valid split;
- leaving it out can produce different rows on another run;
- `42` is an arbitrary example, not a better model setting.

It does not mean row 42, 42 percent, or 42 shuffles.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_remaining, y_train, y_remaining = train_test_split(
    X,
    y,
    test_size=0.40,
    random_state=42,
)

X_validation, X_test, y_validation, y_test = train_test_split(
    X_remaining,
    y_remaining,
    test_size=0.50,
    random_state=42,
)

print("Training rows:", len(X_train))
print("Validation rows:", len(X_validation))
print("Test rows:", len(X_test))


## See What `random_state` Does

The next provided example uses eight visible row labels so the
shuffle is easy to inspect.

It makes three splits:

1. seed 42;
2. seed 42 again;
3. seed 7.

Run the cell and compare the selected test rows. This example is
only a demonstration; our real train, validation, and test sets
above do not change.


In [ ]:
demo_rows = ["A", "B", "C", "D", "E", "F", "G", "H"]

demo_train_42_first, demo_test_42_first = train_test_split(
    demo_rows,
    test_size=0.25,
    random_state=42,
)
demo_train_42_second, demo_test_42_second = train_test_split(
    demo_rows,
    test_size=0.25,
    random_state=42,
)
demo_train_7, demo_test_7 = train_test_split(
    demo_rows,
    test_size=0.25,
    random_state=7,
)

print("Seed 42, first split:", demo_test_42_first)
print("Seed 42, second split:", demo_test_42_second)
print("Seed 7:", demo_test_7)
print(
    "Same seed gives the same test rows:",
    demo_test_42_first == demo_test_42_second,
)
print(
    "Different seed gives different test rows here:",
    demo_test_42_first != demo_test_7,
)


## Mean Absolute Error (MAE)

**MAE** means **Mean Absolute Error**. It answers:

> On average, about how many cups away are the predictions from
> the real sales?

| Day | Real sales | Prediction | Distance |
|---:|---:|---:|---:|
| 1 | 80 | 75 | 5 |
| 2 | 60 | 68 | 8 |
| 3 | 100 | 94 | 6 |

```text
MAE = (5 + 8 + 6) / 3 = 6.3 cups
```

A smaller MAE is better. MAE is measured in cups, not percent.

`mean_absolute_error(real_answers, predictions)` is a
scikit-learn function that performs this calculation for all
rows. We use it so every model is measured with the same rule.

All four models must use the same validation rows and the same
MAE calculation. Otherwise, the comparison would not be fair.


In [ ]:
from sklearn.metrics import mean_absolute_error

actual_sales = [80, 60, 100]
predicted_sales = [75, 68, 94]

example_mae = mean_absolute_error(
    actual_sales,
    predicted_sales,
)

print("Example MAE:", round(example_mae, 1), "cups")


Before continuing, explain:

1. Is a smaller or larger MAE better?
2. If MAE is 8 cups, what does that mean?
3. Why must all four models use the same validation rows?


# Linear Regression

Linear Regression learns one rule that combines the features.
When we change one input and keep the others fixed, its
prediction follows a straight line.

We import `LinearRegression` here because we use it now.


## Task 5 — Train Linear Regression

Complete the three main supervised-learning steps:

1. create the model;
2. use `.fit()` with training data;
3. use `.predict()` with validation features.


In [ ]:
from sklearn.linear_model import LinearRegression

linear_model = None  # TODO: create LinearRegression()

# TODO: fit linear_model with X_train and y_train

linear_predictions = None  # TODO: predict X_validation

linear_mae = mean_absolute_error(
    y_validation,
    linear_predictions,
)

print(
    "Linear Regression validation MAE:",
    round(linear_mae, 2),
    "cups",
)


<details>
<summary>Hint</summary>

```python
model = LinearRegression()
model.fit(training_features, training_target)
predictions = model.predict(validation_features)
```
</details>


## Inside Linear Regression After `.fit()`

`.fit(...)` stores the learned feature coefficients in
`linear_model.coef_` and the starting value in
`linear_model.intercept_`.

`plt.bar(names, values)` draws one bar for each coefficient.
We use it to see whether each feature moves the prediction in a
positive or negative direction.

`plt.axhline(0, ...)` draws a horizontal zero line. We need it
so positive bars and negative bars are easy to distinguish.

The coefficients use different feature units. Today, compare
their directions—not their heights as feature importance.


In [ ]:
print(
    "Learned intercept:",
    round(linear_model.intercept_, 2),
)
print("Feature order:", feature_columns)
print(
    "Learned coefficients:",
    [
        round(float(linear_model.coef_[0]), 2),
        round(float(linear_model.coef_[1]), 2),
        round(float(linear_model.coef_[2]), 2),
    ],
)

plt.bar(
    feature_columns,
    linear_model.coef_,
    color=["#42A5F5", "#66BB6A", "#EF5350"],
)
plt.axhline(0, color="black", linewidth=1)
plt.ylabel("Learned coefficient")
plt.title("What Linear Regression stored during .fit()")
plt.show()


**Read the coefficients:**

1. Which features have positive coefficients?
2. Which feature has a negative coefficient?
3. Were these learned values available before `.fit()`?


## One Linear Regression Prediction, Step by Step

Linear Regression does not expose a list of training epochs
here. Its `.fit(...)` calculates the coefficients and intercept
directly. After that, `.predict(...)` performs this real
calculation for each row:

```text
intercept
+ temperature × temperature coefficient
+ weekend × weekend coefficient
+ rain × rain coefficient
= prediction
```

The next cell uses an 80°F rainy weekend so all three feature
contributions are visible. The provided bar chart shows the
four numbers that are added together.


In [ ]:
linear_example_day = pd.DataFrame({
    "temperature_f": [80],
    "weekend": [1],
    "rain": [1],
})

intercept_part = linear_model.intercept_
temperature_part = (
    80 * linear_model.coef_[0]
)
weekend_part = 1 * linear_model.coef_[1]
rain_part = 1 * linear_model.coef_[2]

manual_linear_prediction = (
    intercept_part
    + temperature_part
    + weekend_part
    + rain_part
)
sklearn_linear_prediction = linear_model.predict(
    linear_example_day
)[0]

print("Intercept:", round(intercept_part, 2))
print(
    "Temperature contribution:",
    round(temperature_part, 2),
)
print(
    "Weekend contribution:",
    round(weekend_part, 2),
)
print("Rain contribution:", round(rain_part, 2))
print(
    "Manual total:",
    round(manual_linear_prediction, 2),
)
print(
    "Model prediction:",
    round(sklearn_linear_prediction, 2),
)

plt.bar(
    ["Intercept", "Temperature", "Weekend", "Rain"],
    [
        intercept_part,
        temperature_part,
        weekend_part,
        rain_part,
    ],
    color=[
        "#78909C",
        "#42A5F5",
        "#66BB6A",
        "#EF5350",
    ],
)
plt.axhline(0, color="black", linewidth=1)
plt.ylabel("Cups added to the prediction")
plt.title("Four parts of one Linear Regression prediction")
plt.show()


**Follow the calculation:**

1. Which contribution increases the prediction the most?
2. Which contributions decrease it?
3. Does the manual total match `.predict(...)`?


## See the Learned Linear Rule

The following chart asks the model about many dry weekdays.
We change only temperature and keep:

```text
weekend = 0
rain = 0
```

`range(55, 96)` creates temperatures from 55 through 95.
`pd.DataFrame(...)` organizes those temperatures and the fixed
weekend/rain values into the same table shape the model expects.
We need these new input rows so `.predict(...)` can draw the
learned rule across many temperatures.

The plotting code is provided. Focus on the blue points and the
red prediction line.


In [ ]:
temperature_values = list(range(55, 96))
dry_weekdays = pd.DataFrame({
    "temperature_f": temperature_values,
    "weekend": [0] * len(temperature_values),
    "rain": [0] * len(temperature_values),
})

linear_curve = linear_model.predict(dry_weekdays)
dry_weekday_rows = (
    (X_train["weekend"] == 0)
    & (X_train["rain"] == 0)
)

plt.scatter(
    X_train.loc[dry_weekday_rows, "temperature_f"],
    y_train.loc[dry_weekday_rows],
    color="#1976D2",
    label="Real dry weekdays",
)
plt.plot(
    temperature_values,
    linear_curve,
    color="#E53935",
    linewidth=3,
    label="Linear Regression prediction",
)
plt.xlabel("Temperature (°F)")
plt.ylabel("Cups sold")
plt.title("The rule learned by Linear Regression")
plt.legend()
plt.show()


**Look at the chart:**

- Does the line pass through every real point?
- What happens to predicted sales when temperature increases?
- Why did we keep `weekend` and `rain` fixed?


# Decision Tree

A Decision Tree learns a series of questions:

```text
Is temperature below a value?
├── Yes → ask another question or make a prediction
└── No  → ask another question or make a prediction
```

`max_depth` limits how many levels of questions the tree can
use. A small limit keeps the tree easier to read.


## Task 6 — Train a Decision Tree

`DecisionTreeRegressor(...)` creates a Decision Tree that
predicts numbers. We use the regressor version because
`cups_sold` is a number, not a category.

Its important settings today are:

- `max_depth=3`: use at most three levels of questions;
- `min_samples_leaf=3`: keep at least three training rows in a
  final group;
- `random_state=42`: make the result repeatable.


In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree_model = None  # TODO: create the model shown in the hint

# TODO: fit tree_model with X_train and y_train

tree_predictions = None  # TODO: predict X_validation
tree_mae = mean_absolute_error(
    y_validation,
    tree_predictions,
)

print(
    "Decision Tree validation MAE:",
    round(tree_mae, 2),
    "cups",
)


<details>
<summary>Hint</summary>

Create the model with:

```python
DecisionTreeRegressor(
    max_depth=3,
    min_samples_leaf=3,
    random_state=42,
)
```

Then use the same `.fit()` and `.predict()` pattern as Linear
Regression.
</details>


## See the Questions Learned by the Tree

`plot_tree(tree_model, ...)` draws the questions and predictions
stored inside a trained Decision Tree. We use it because the
picture lets us follow how one input row reaches a prediction.
The remaining arguments control labels and appearance; they are
provided and do not need to be memorized.

For this fixed training set, the tree grows from top to bottom:

1. it first chooses `temperature_f <= 75` as the root question;
2. it then adds questions for the two resulting groups;
3. it continues until the `max_depth=3` limit or another stopping
   rule is reached;
4. each final leaf stores the average target for its group.

The chart is read from top to bottom.

- Each box contains a question or prediction.
- The left branch means the question is true.
- The right branch means the question is false.


In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(16, 7))
plot_tree(
    tree_model,
    feature_names=feature_columns,
    filled=True,
    rounded=True,
    impurity=False,
    fontsize=9,
)
plt.title("Questions learned by the Decision Tree")
plt.show()


## Follow One Day Through the Tree

`pd.DataFrame(...)` creates one new input row in the same format
used for training. `.predict(...)` sends that row through the
learned questions.

`display(table)` shows the input as a labeled table in Colab.
We use it so you can read each feature value before tracing the
path in the tree picture above.


In [ ]:
tree_example_day = pd.DataFrame({
    "temperature_f": [80],
    "weekend": [0],
    "rain": [0],
})
tree_example_prediction = tree_model.predict(
    tree_example_day
)[0]

display(tree_example_day)
print(
    "Decision Tree prediction:",
    round(tree_example_prediction, 1),
    "cups",
)


### The Actual Path for This Row

```text
Step 1: Is temperature_f <= 75?
        80 <= 75 is False → move right

Step 2: Is temperature_f <= 88?
        80 <= 88 is True → move left

Step 3: Is rain <= 0.5?
        0 <= 0.5 is True → move left

Final leaf: predict about 62.8 cups
```

These questions and thresholds were learned during `.fit(...)`.
`.predict(...)` follows them in this order.


**Read the tree:**

1. What is the first question?
2. For the 80°F dry weekday, do you move left or right?
3. Which final prediction does the row reach?
4. Does every final group predict the same number?
5. What might happen if `max_depth` becomes very large?


# K-Nearest Neighbors (KNN)

To predict a new day, KNN:

1. finds similar training days;
2. selects the nearest `K` days;
3. averages their known sales.

With `K=5`, the model uses five nearby training examples.


## Why Scaling Is Provided

KNN measures distance. Temperature values such as `80` are much
larger than weekend and rain values such as `0` or `1`.

`StandardScaler()` puts the input columns on similar numeric
scales. We need it so temperature does not control the distance
calculation only because its numbers are larger.

`KNeighborsRegressor(n_neighbors=5)` creates a KNN model that
predicts a number by averaging five nearby training answers.

`make_pipeline(step_1, step_2)` connects tools in order. Here it
applies `StandardScaler()` first and KNN second during both
`.fit(...)` and `.predict(...)`. We use the pipeline so we do
not accidentally scale training and validation data in
different ways.

You do not need to memorize the settings today, but you should
know why each tool is present.


## Task 7 — Train KNN

We import `StandardScaler`, `make_pipeline`, and
`KNeighborsRegressor` here because this is their first use.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsRegressor

number_of_neighbors = None  # TODO: use 5

knn_model = make_pipeline(
    StandardScaler(),
    KNeighborsRegressor(
        n_neighbors=number_of_neighbors,
    ),
)

# TODO: fit knn_model with X_train and y_train

knn_predictions = None  # TODO: predict X_validation
knn_mae = mean_absolute_error(
    y_validation,
    knn_predictions,
)

print(
    "KNN validation MAE:",
    round(knn_mae, 2),
    "cups",
)


## How K Changes KNN: Compare K = 5 and K = 15

**K is the number of nearby training rows whose known answers
are averaged for one prediction.**

- `K=5` uses a smaller, more local group.
- `K=15` uses a larger, broader group.

The next provided cell compares both settings using the same
training rows and the same validation rows. It also draws both
prediction curves on one chart:

- `knn_model` is the `K=5` model from Task 7;
- `KNeighborsRegressor(n_neighbors=15)` creates the second
  model;
- `mean_absolute_error(...)` measures both models in the same
  way;
- the test data remains closed during this comparison.

All plotting tools in this cell were introduced earlier. Focus
on how the two prediction curves differ.


In [ ]:
knn_5_model = knn_model

knn_15_model = make_pipeline(
    StandardScaler(),
    KNeighborsRegressor(n_neighbors=15),
)
knn_15_model.fit(X_train, y_train)

knn_5_validation_predictions = knn_5_model.predict(
    X_validation
)
knn_15_validation_predictions = knn_15_model.predict(
    X_validation
)
knn_5_mae = mean_absolute_error(
    y_validation,
    knn_5_validation_predictions,
)
knn_15_mae = mean_absolute_error(
    y_validation,
    knn_15_validation_predictions,
)

print(
    "K = 5 validation MAE:",
    round(knn_5_mae, 2),
    "cups",
)
print(
    "K = 15 validation MAE:",
    round(knn_15_mae, 2),
    "cups",
)

knn_5_curve = knn_5_model.predict(dry_weekdays)
knn_15_curve = knn_15_model.predict(dry_weekdays)

plt.scatter(
    X_train.loc[dry_weekday_rows, "temperature_f"],
    y_train.loc[dry_weekday_rows],
    color="#B0BEC5",
    alpha=0.65,
    label="Real dry weekdays",
)
plt.plot(
    temperature_values,
    knn_5_curve,
    color="#FB8C00",
    linewidth=3,
    label="K = 5",
)
plt.plot(
    temperature_values,
    knn_15_curve,
    color="#00897B",
    linewidth=3,
    label="K = 15",
)
plt.xlabel("Temperature (°F)")
plt.ylabel("Predicted cups sold")
plt.title("KNN predictions change when K changes")
plt.legend()
plt.show()


### What Is the Difference?

On this fixed data split:

- `K=5` has validation MAE of about **8.77 cups**;
- `K=15` has validation MAE of about **11.78 cups**.

`K=5` reacts more to nearby training examples, so its curve
changes more. `K=15` averages a wider group, so its curve is
smoother, but some of those 15 days may be less similar to the
new day.

### How Do We Choose K?

`K` is a model setting chosen before `.fit(...)`. It is also
called a **hyperparameter**.

- If K is very small, one unusual training row can change the
  prediction too much.
- If K is very large, many less-similar rows are mixed together
  and useful local differences may disappear.
- We can train several candidate KNN models on the training
  data and compare them on the validation data.
- We do **not** use the test data to choose K. Test data stays
  closed until the final check.

For today's internal-process visualization, we continue with
`K=5` because five neighbors are easy to see and average. This
does not mean that `K=5` is best for every dataset.


## KNN Visualization: Five Neighbors and Their Average

The provided code asks KNN to predict a dry weekday at 80°F.
Internally, the model finds five nearby training days and
averages their known sales.

KNN is different from the other models: `.fit(...)` mainly
stores the scaled training examples. The neighbor search happens
when `.predict(...)` receives a new row.

The provided visualization uses:

- `knn_model.named_steps[...]` to access the fitted scaler and
  KNN stored inside the pipeline;
- `.transform(example_day)` to apply the already learned scale
  to the new row;
- `.kneighbors(...)` to return the distances and row positions
  of the five nearest training examples;
- `pd.concat(..., axis=1)` to place their features and known
  sales in one table;
- `plt.axvline(80, ...)` to mark the new day's temperature on
  the chart;
- `plt.xlim(...)` and `plt.ylim(...)` to zoom in so the selected
  neighbors are easy to see.

We use these tools only to reveal what KNN is doing internally.
Run the provided cell and look for:

- **large orange circles:** the five selected neighbors;
- **a large red star:** the new day's predicted sales;
- **five orange bars:** the known answers being averaged;
- **a red horizontal line:** their average and the KNN
  prediction.

You do not need to memorize the inspection code.


In [ ]:
example_day = pd.DataFrame({
    "temperature_f": [80],
    "weekend": [0],
    "rain": [0],
})

fitted_scaler = knn_model.named_steps["standardscaler"]
fitted_knn = knn_model.named_steps["kneighborsregressor"]

scaled_example = fitted_scaler.transform(example_day)
distances, neighbor_positions = fitted_knn.kneighbors(
    scaled_example
)

neighbor_features = X_train.iloc[neighbor_positions[0]]
neighbor_answers = y_train.iloc[neighbor_positions[0]]
neighbor_table = pd.concat(
    [neighbor_features, neighbor_answers],
    axis=1,
)
neighbor_table["scaled_distance"] = distances[0]

neighbor_average = neighbor_table["cups_sold"].mean()
example_prediction = knn_model.predict(example_day)[0]

print("Step 1 — New raw input:")
display(example_day)
print(
    "Step 2 — Scaled input:",
    [
        round(float(scaled_example[0][0]), 2),
        round(float(scaled_example[0][1]), 2),
        round(float(scaled_example[0][2]), 2),
    ],
)
print(
    "Step 3 — Five smallest scaled distances "
    "and their known answers:"
)
display(neighbor_table)
print(
    "Step 4 — Average the five answers:",
    round(neighbor_average, 1),
    "cups",
)
print(
    "Step 5 — KNN prediction:",
    round(example_prediction, 1),
    "cups",
)

dry_weekday_training = (
    (X_train["weekend"] == 0)
    & (X_train["rain"] == 0)
)
plt.scatter(
    X_train.loc[
        dry_weekday_training,
        "temperature_f",
    ],
    y_train.loc[dry_weekday_training],
    color="#B0BEC5",
    alpha=0.45,
    s=45,
    label="Other dry weekdays",
)
plt.scatter(
    neighbor_table["temperature_f"],
    neighbor_table["cups_sold"],
    color="#FB8C00",
    edgecolor="black",
    linewidth=1.5,
    s=240,
    zorder=3,
    label="Five selected neighbors",
)
plt.scatter(
    [80],
    [example_prediction],
    color="#D32F2F",
    edgecolor="black",
    marker="*",
    s=500,
    zorder=4,
    label=(
        "New prediction: "
        f"{example_prediction:.1f} cups"
    ),
)
plt.axvline(
    80,
    color="#212121",
    linestyle="--",
    label="New day: 80°F",
)
plt.xlim(77.5, 82.5)
plt.ylim(49, 71)
plt.xlabel("Temperature (°F)")
plt.ylabel("Cups sold")
plt.title("KNN zoom: five neighbors and the new prediction")
plt.legend()
plt.show()

neighbor_labels = [
    "Neighbor 1",
    "Neighbor 2",
    "Neighbor 3",
    "Neighbor 4",
    "Neighbor 5",
]
plt.bar(
    neighbor_labels,
    neighbor_table["cups_sold"],
    color="#FB8C00",
    edgecolor="black",
)
plt.axhline(
    example_prediction,
    color="#D32F2F",
    linewidth=3,
    label=(
        "Average = prediction = "
        f"{example_prediction:.1f}"
    ),
)
plt.ylabel("Known cups sold")
plt.title("KNN prediction is the average of five answers")
plt.legend()
plt.show()


Think about the result:

1. Which five training rows were selected?
2. What do the five orange bars represent?
3. Where is the red star relative to the five neighbors?
4. Does the red average line match the KNN prediction?
5. Did KNN learn one equation like Linear Regression?


# A Small Neural Network

A neural network connects small calculating units called
neurons.

```text
temperature ─┐
weekend ─────┼→ 8 hidden neurons → predicted cups
rain ────────┘
```

The hidden neurons let the model combine the features in more
flexible ways. Today we use scikit-learn so the main steps remain
`.fit()` and `.predict()`.

We do not use PyTorch or write the training calculations
ourselves today. `MLPRegressor` provides the neural-network
model while keeping the familiar `.fit()` and `.predict()`
pattern.


## Task 8 — Train the Neural Network

`MLPRegressor(...)` creates a small multilayer neural network
for predicting numbers. We use it because the target,
`cups_sold`, is numeric and we want to see a model that can
learn a more flexible rule.

In today's model:

- `hidden_layer_sizes=(8,)` means one hidden layer with eight
  neurons;
- `solver="adam"` chooses a step-by-step training procedure
  that records a loss value after each iteration;
- `learning_rate_init=0.02` controls the size of each weight
  adjustment;
- `max_iter=2500` limits how many training iterations it may
  use;
- `random_state=42` makes the result repeatable.

We use `StandardScaler()` again because neural-network training
works more reliably when input columns have similar scales.
`make_pipeline(...)` applies scaling before the neural network.

The training details are provided so you can focus on the common
supervised-learning process.


In [ ]:
from sklearn.neural_network import MLPRegressor

hidden_neurons = None  # TODO: use 8

neural_network_model = make_pipeline(
    StandardScaler(),
    MLPRegressor(
        hidden_layer_sizes=(hidden_neurons,),
        solver="adam",
        learning_rate_init=0.02,
        max_iter=2500,
        random_state=42,
    ),
)

# TODO: fit neural_network_model with X_train and y_train

neural_network_predictions = None  # TODO: predict X_validation
neural_network_mae = mean_absolute_error(
    y_validation,
    neural_network_predictions,
)

print(
    "Neural Network validation MAE:",
    round(neural_network_mae, 2),
    "cups",
)


## Inside the Neural Network During and After `.fit()`

Before learning this dataset, the network has not adjusted its
connections to these training rows. During `.fit(...)`, the
training procedure repeatedly changes connection weights to
reduce training error. The final weights are stored in
`MLPRegressor.coefs_`.

With `solver="adam"`, `MLPRegressor.loss_curve_` records one
internal training-loss value after each iteration.
`plt.plot(iterations, loss_values)` draws how that value changes.
We use this curve to see the repeated adjustment process rather
than only the final result.

The code also displays six snapshots from the loss history:
after iterations 1, 50, 100, 200, 500, and the final iteration.
This makes the sequence visible as numbers as well as a curve.

Training loss is not validation MAE. A falling training-loss
curve shows that the network is fitting the training rows;
validation MAE still decides whether it works well on unseen
rows.

`coefs_[0]` contains weights from the three inputs to the eight
hidden neurons:

```text
3 input rows × 8 hidden-neuron columns
```

`plt.imshow(matrix, ...)` turns the weight matrix into a color
image. `plt.colorbar(...)` explains the color scale, and
`plt.yticks(...)` labels the three input rows. We use a heatmap
because it makes many learned connections visible at once.

A single weight does not explain the whole prediction. The
network combines all of these weights and another set of
hidden-to-output weights.


In [ ]:
fitted_network = neural_network_model.named_steps[
    "mlpregressor"
]
training_loss_curve = fitted_network.loss_curve_
input_to_hidden_weights = fitted_network.coefs_[0]
progress_iterations = [
    1,
    50,
    100,
    200,
    500,
    len(training_loss_curve),
]
progress_losses = [
    round(training_loss_curve[0], 2),
    round(training_loss_curve[49], 2),
    round(training_loss_curve[99], 2),
    round(training_loss_curve[199], 2),
    round(training_loss_curve[499], 2),
    round(training_loss_curve[-1], 2),
]
training_progress = pd.DataFrame({
    "iteration": progress_iterations,
    "training_loss": progress_losses,
})

print(
    "Recorded training iterations:",
    len(training_loss_curve),
)
print(
    "First training loss:",
    round(training_loss_curve[0], 2),
)
print(
    "Final training loss:",
    round(training_loss_curve[-1], 2),
)
print(
    "Input-to-hidden weight shape:",
    input_to_hidden_weights.shape,
)
print("Selected moments during training:")
display(training_progress)

plt.plot(
    range(1, len(training_loss_curve) + 1),
    training_loss_curve,
    color="#8E24AA",
    linewidth=2,
)
plt.xlabel("Training iteration")
plt.ylabel("Internal training loss")
plt.title(
    "Neural-network loss changes during .fit()"
)
plt.show()

weight_image = plt.imshow(
    input_to_hidden_weights,
    aspect="auto",
    cmap="coolwarm",
)
plt.colorbar(
    weight_image,
    label="Learned weight",
)
plt.yticks(
    range(len(feature_columns)),
    feature_columns,
)
plt.xlabel("Hidden neuron number")
plt.ylabel("Input feature")
plt.title("Weights learned during neural-network .fit()")
plt.show()


**Read the heatmap:**

1. Does training loss generally decrease during `.fit()`?
2. Why is training loss different from validation MAE?
3. Why are there three heatmap rows?
4. Why are there eight columns?
5. Are all connection weights identical?


# Compare the Four Models

The next chart shows what each trained model predicts for dry
weekdays while temperature changes.

Every model:

- learned from the same training rows;
- received the same three features;
- is evaluated on the same validation rows.

This cell introduces several plotting helpers because one chart
must contain four smaller charts:

| New plotting tool | What it does | Why we use it |
|---|---|---|
| `plt.subplots(2, 2)` | creates a figure with four chart areas | show four models together |
| `axes.ravel()` | puts the four chart areas in one sequence | visit them with one loop |
| `zip(axes.ravel(), comparison_models)` | pairs each chart area with one model | draw the correct model in each area |
| `axis.scatter(...)` | draws the real training points | compare predictions with data |
| `axis.plot(...)` | draws one model's prediction curve | see the learned shape |
| `axis.set_...(...)` and `axis.legend()` | add labels and a legend | make each chart readable |
| `figure.suptitle(...)` | adds one title above all four charts | describe the whole figure |
| `plt.tight_layout()` | adjusts spacing | keep labels from overlapping |

The plotting loop is provided. Run it, but do not try to
memorize every plotting instruction. Focus on the four learned
shapes.


In [ ]:
comparison_models = [
    ("Linear Regression", linear_model, "#E53935"),
    ("Decision Tree", tree_model, "#43A047"),
    ("KNN", knn_model, "#FB8C00"),
    ("Neural Network", neural_network_model, "#8E24AA"),
]

figure, axes = plt.subplots(
    2,
    2,
    figsize=(13, 9),
    sharex=True,
    sharey=True,
)

for axis, (name, model, color) in zip(
    axes.ravel(),
    comparison_models,
):
    curve = model.predict(dry_weekdays)
    axis.scatter(
        X_train.loc[dry_weekday_rows, "temperature_f"],
        y_train.loc[dry_weekday_rows],
        color="#1976D2",
        alpha=0.75,
        label="Real training days",
    )
    axis.plot(
        temperature_values,
        curve,
        color=color,
        linewidth=3,
        label="Predictions",
    )
    axis.set_title(name)
    axis.set_xlabel("Temperature (°F)")
    axis.set_ylabel("Cups sold")
    axis.legend()

figure.suptitle(
    "Same data, different learned prediction rules",
    fontsize=16,
)
plt.tight_layout()
plt.show()


**Compare the shapes:**

1. Which model makes a straight line?
2. Which model makes steps?
3. Which models can learn more flexible shapes?
4. Did the neural network need a strongly curved shape here?
5. Does a more complicated shape automatically mean a better
   validation MAE?


## Task 9 — Compare Validation MAE

A fair comparison uses the same validation rows and the same
metric. The smallest validation MAE wins this comparison.

This is why Day 2 asks us to choose a model: today's goal is to
compare different algorithms. Day 1 used Linear Regression to
introduce the learning process.

Choose from the validation results—not from which model sounds
newest or most complicated.

The next cell builds and displays the comparison:

- `pd.DataFrame(...)` puts the four names and MAE values into a
  table;
- `.sort_values("validation_mae_cups")` orders the rows from
  the smallest MAE to the largest, so the best result appears
  first;
- `.round(2)` shows two decimal places without retraining or
  changing any model;
- `display(...)` shows the finished table clearly in Colab.


In [ ]:
validation_results = pd.DataFrame({
    "model": [
        "Linear Regression",
        "Decision Tree",
        "KNN",
        "Neural Network",
    ],
    "validation_mae_cups": [
        linear_mae,
        tree_mae,
        knn_mae,
        neural_network_mae,
    ],
}).sort_values("validation_mae_cups")

display(validation_results.round(2))


Record your choice before using test data:

> We selected __________. Its validation MAE was __________
> cups. We selected it because ____________________________.


In [ ]:
chosen_model_name = ""  # TODO: enter the winning model name
chosen_model = None     # TODO: enter its model variable

print("Chosen model:", chosen_model_name)


## Task 10 — Final Test

Use the test set only after choosing the model with
validation MAE.

Complete:

```text
We selected __________.
Its validation MAE was __________ cups.
Its final test MAE was __________ cups.
```


In [ ]:
test_predictions = None  # TODO: predict X_test
test_mae = None          # TODO: compare y_test and predictions

print("Final test MAE:", round(test_mae, 2), "cups")


# Review

Answer in complete sentences:

1. What makes this supervised learning?
2. What number did all four models predict?
3. What stayed the same when the model changed?
4. How does a Decision Tree make predictions?
5. What does `K=5` mean in KNN?
6. What is the hidden layer in the neural network?
7. What does the final test MAE mean in everyday language?
8. Why should test data not be used repeatedly to choose models?
9. Why did the most complicated model not automatically win?


# Optional Extra Tasks

Choose one:

- Change the Decision Tree's `max_depth` and compare validation
  MAE.
- Change KNN's `number_of_neighbors`.
- Change the Neural Network's `hidden_neurons`.
- Predict sales for an 80°F rainy weekend.
- Make a bar chart of the four validation MAE values.

**Think carefully:** A more complicated model can follow more
patterns. Why does that not guarantee a better result on new
data?
